In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import os

fake = Faker('ru_RU')
np.random.seed(42)
random.seed(42)

print("🚀 Генерация данных для строительной компании (УрФО)")
print("="*50)

# ==================== ИЗМЕРЕНИЯ ====================

# 1. Менеджеры (живые фамилии)
dim_manager = pd.DataFrame({
    'Manager_ID': [1, 2, 3, 4, 5],
    'Manager_Name': ['Алексеев Д.М.', 'Васнецова Е.П.', 'Крылов С.И.', 'Морозова О.В.', 'Румянцев А.Г.'],
    'Department': ['Управление проектами'] * 5,
    'Specialization': ['Теплосети', 'Водоснабжение', 'Котельные', 'Насосные станции', 'Теплосети'],
    'Experience_Years': [12, 8, 15, 10, 7]
})

# 2. Статусы проектов
dim_status = pd.DataFrame({
    'Status_ID': [1, 2, 3, 4, 5],
    'Status_Name': ['Планирование', 'В работе', 'На согласовании', 'Завершен', 'Проблема'],
    'Status_Group': ['Активные', 'Активные', 'Активные', 'Завершенные', 'Рисковые'],
    'Status_Color': ['#FFC107', '#17A2B8', '#007BFF', '#28A745', '#DC3545']
})

# 3. Приоритеты
dim_priority = pd.DataFrame({
    'Priority_ID': [1, 2, 3],
    'Priority_Name': ['Высокий', 'Средний', 'Низкий'],
    'Priority_Level': [3, 2, 1],
    'Priority_Color': ['#DC3545', '#FFC107', '#28A745']
})

# 4. Типы проектов (строительные)
dim_project_type = pd.DataFrame({
    'Type_ID': [1, 2, 3, 4],
    'Project_Type': ['Теплосети', 'Водоснабжение', 'Насосные станции', 'Котельные'],
    'Type_Category': ['Инженерные сети', 'Инженерные сети', 'Оборудование', 'Оборудование'],
    'Avg_Duration_Months': [8, 6, 5, 9]
})

# 5. Регионы УрФО (только 4 области)
dim_region = pd.DataFrame({
    'Region_ID': [1, 2, 3, 4],
    'Region_Name': ['Екатеринбург', 'Челябинск', 'Тюмень', 'Курган'],
    'Oblast': ['Свердловская', 'Челябинская', 'Тюменская', 'Курганская'],
    'Population': [1500000, 1200000, 850000, 300000]
})

print("✅ Измерения созданы")

# ==================== ТАБЛИЦА ФАКТОВ (40 тендеров) ====================

def generate_tenders(n=40):
    tenders = []
    
    # Список названий для реалистичности
    prefixes = ['Реконструкция', 'Строительство', 'Модернизация', 'Капремонт', 'Прокладка']
    objects = ['теплосети', 'котельной', 'водовода', 'насосной станции', 'теплотрассы']
    locations = ['Центральная', 'Северная', 'Южная', 'Восточная', 'Западная', 'Промзона', 'мкр Южный', 'мкр Северный']
    
    for i in range(1, n+1):
        # Выбираем тип проекта
        project_type = random.choice(dim_project_type['Project_Type'].tolist())
        
        # Генерируем название
        if i <= len(locations):
            tender_name = f"{random.choice(prefixes)} {project_type.lower()} '{locations[i-1]}'"
        else:
            tender_name = f"{random.choice(prefixes)} {project_type.lower()} в {random.choice(['районе', 'квартале', 'промзоне'])}"
        
        # Менеджер (с привязкой к специализации)
        if project_type == 'Теплосети':
            manager_id = random.choice([1, 5])  # Алексеев или Румянцев
        elif project_type == 'Водоснабжение':
            manager_id = 2  # Васнецова
        elif project_type == 'Котельные':
            manager_id = 3  # Крылов
        else:  # Насосные станции
            manager_id = 4  # Морозова
        
        # Статус
        status_weights = [0.15, 0.4, 0.1, 0.2, 0.15]  # Планирование, В работе, На согласовании, Завершен, Проблема
        status_id = random.choices(range(1, 6), weights=status_weights)[0]
        
        # Приоритет (веса в зависимости от статуса)
        if status_id == 5:  # Проблема
            priority_weights = [0.7, 0.2, 0.1]  # Чаще высокий
        else:
            priority_weights = [0.3, 0.5, 0.2]  # Обычное распределение
        priority_id = random.choices(range(1, 4), weights=priority_weights)[0]
        
        # Даты
        if status_id == 4:  # Завершен
            start_date = fake.date_between(start_date='-180d', end_date='-90d')
            deadline_delta = random.randint(60, 150)
            deadline_date = start_date + timedelta(days=deadline_delta)
            fact_end_date = start_date + timedelta(days=random.randint(50, deadline_delta-5))
            progress = 100
        else:
            start_date = fake.date_between(start_date='-90d', end_date='+30d')
            deadline_delta = random.randint(60, 180)
            deadline_date = start_date + timedelta(days=deadline_delta)
            fact_end_date = None
            
            # Прогресс
            if start_date > datetime.now().date():
                progress = 0
            else:
                total_days = (deadline_date - start_date).days
                days_passed = (datetime.now().date() - start_date).days
                progress = min(100, round((days_passed / total_days) * 100))
        
        # Бюджет (от 2 до 30 млн)
        budget_planned = round(random.uniform(2000000, 30000000), -4)
        
        # Отклонения в зависимости от статуса
        if status_id == 5:  # Проблема
            variance = random.uniform(0.1, 0.3)  # Перерасход 10-30%
        elif status_id == 4:  # Завершен
            variance = random.uniform(-0.05, 0.1)  # Экономия или небольшой перерасход
        else:
            variance = random.uniform(-0.05, 0.15)
            
        budget_current = round(budget_planned * (1 + variance * 0.7), -4)
        budget_forecast = round(budget_current * random.uniform(0.95, 1.2), -4)
        
        # Регион
        region_id = random.randint(1, 4)
        
        tenders.append({
            'Tender_ID': i,
            'Tender_Name': tender_name,
            'Project_Type': project_type,
            'Region_ID': region_id,
            'Manager_ID': manager_id,
            'Status_ID': status_id,
            'Priority_ID': priority_id,
            'Budget_Planned': budget_planned,
            'Budget_Current': budget_current,
            'Budget_Forecast': budget_forecast,
            'Start_Date': start_date,
            'Deadline_Date': deadline_date,
            'Fact_End_Date': fact_end_date,
            'Progress_Percent': progress
        })
    
    return pd.DataFrame(tenders)

# Генерируем 40 тендеров
fact_tenders = generate_tenders(40)
print(f"✅ Создано {len(fact_tenders)} тендеров")

# ==================== СУБПОДРЯДЧИКИ ====================

# Список субподрядчиков
subcontractors_list = [
    'УралТеплоМонтаж', 'ЭлектроСеть', 'Котельный завод', 'АвтоматикаУрал',
    'ВодоканалСтрой', 'СпецТехника', 'ТрубопроводСервис', 'СеверТеплоСтрой',
    'НасосСнаб', 'ТеплоСетиУрала', 'БетонСтрой', 'ГидроМонтаж',
    'УралЭнерго', 'СтройМеханизация', 'ТеплоИзоляция'
]

work_types = ['Монтаж труб', 'Электрика', 'Оборудование', 'Автоматизация',
              'Земляные работы', 'Бетонные работы', 'Изоляция', 'Прокладка труб']

def generate_subcontractors(tenders_df, n_relations=45):
    records = []
    record_id = 101
    
    for _, tender in tenders_df.iterrows():
        # У каждого тендера 1-3 субподрядчика
        n_subs = random.randint(1, 3)
        
        for _ in range(n_subs):
            if len(records) >= n_relations:
                break
                
            subcontractor = random.choice(subcontractors_list)
            work_type = random.choice(work_types)
            
            # Сумма контракта (от 300к до 8 млн)
            contract_amount = round(random.uniform(300000, 8000000), -3)
            
            # Даты работ (в рамках тендера)
            start_date = fake.date_between(
                start_date=tender['Start_Date'],
                end_date=tender['Deadline_Date'] - timedelta(days=30)
            )
            end_date = fake.date_between(
                start_date=start_date,
                end_date=tender['Deadline_Date']
            )
            
            # Качество и задержки
            quality_score = round(random.uniform(3.0, 5.0), 1)
            
            # 30% субподрядчиков с задержками
            if random.random() < 0.3:
                delay_days = random.randint(5, 25)
            else:
                delay_days = 0
            
            records.append({
                'Subcontract_ID': record_id,
                'Tender_ID': tender['Tender_ID'],
                'Subcontractor_Name': subcontractor,
                'Contract_Amount': contract_amount,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Work_Type': work_type,
                'Quality_Score': quality_score,
                'Delay_Days': delay_days
            })
            record_id += 1
            
            if len(records) >= n_relations:
                break
    
    return pd.DataFrame(records)

fact_subcontractors = generate_subcontractors(fact_tenders, 50)
print(f"✅ Создано {len(fact_subcontractors)} связей с субподрядчиками")

# ==================== СОХРАНЕНИЕ ====================

print("\n💾 Сохранение файлов...")

# Измерения
dim_manager.to_csv('dim_manager.csv', index=False, encoding='utf-8-sig')
dim_status.to_csv('dim_status.csv', index=False, encoding='utf-8-sig')
dim_priority.to_csv('dim_priority.csv', index=False, encoding='utf-8-sig')
dim_project_type.to_csv('dim_project_type.csv', index=False, encoding='utf-8-sig')
dim_region.to_csv('dim_region.csv', index=False, encoding='utf-8-sig')

# Факты
fact_tenders.to_csv('fact_tenders.csv', index=False, encoding='utf-8-sig')
fact_subcontractors.to_csv('fact_subcontractors.csv', index=False, encoding='utf-8-sig')

# Сохраняем в Excel для удобства
with pd.ExcelWriter('tender_data_ural.xlsx', engine='openpyxl') as writer:
    dim_manager.to_excel(writer, sheet_name='dim_manager', index=False)
    dim_status.to_excel(writer, sheet_name='dim_status', index=False)
    dim_priority.to_excel(writer, sheet_name='dim_priority', index=False)
    dim_project_type.to_excel(writer, sheet_name='dim_project_type', index=False)
    dim_region.to_excel(writer, sheet_name='dim_region', index=False)
    fact_tenders.to_excel(writer, sheet_name='fact_tenders', index=False)
    fact_subcontractors.to_excel(writer, sheet_name='fact_subcontractors', index=False)

print("\n" + "="*50)
print("🎉 ГЕНЕРАЦИЯ ЗАВЕРШЕНА!")
print("="*50)
print(f"\n📊 Статистика:")
print(f"  - Тендеров: {len(fact_tenders)}")
print(f"  - Менеджеров: {len(dim_manager)}")
print(f"  - Субподрядчиков: {len(fact_subcontractors)} записей")
print(f"  - Регионов: {len(dim_region)}")
print(f"\n📁 Созданы файлы:")
print("  - dim_manager.csv")
print("  - dim_status.csv") 
print("  - dim_priority.csv")
print("  - dim_project_type.csv")
print("  - dim_region.csv")
print("  - fact_tenders.csv")
print("  - fact_subcontractors.csv")
print("  - tender_data_ural.xlsx")


🚀 Генерация данных для строительной компании (УрФО)
✅ Измерения созданы
✅ Создано 40 тендеров
✅ Создано 50 связей с субподрядчиками

💾 Сохранение файлов...

🎉 ГЕНЕРАЦИЯ ЗАВЕРШЕНА!

📊 Статистика:
  - Тендеров: 40
  - Менеджеров: 5
  - Субподрядчиков: 50 записей
  - Регионов: 4

📁 Созданы файлы:
  - dim_manager.csv
  - dim_status.csv
  - dim_priority.csv
  - dim_project_type.csv
  - dim_region.csv
  - fact_tenders.csv
  - fact_subcontractors.csv
  - tender_data_ural.xlsx
